# 26. Attention 기초 복습

이 노트북은 `25_CNN에서_Transformer로.ipynb` 다음 단계로, Vision Transformer를 이해하는 데 필요한 attention의 기본 계산을 복습합니다.

attention은 각 token이 다른 token을 얼마나 참고할지 가중치로 정하는 연산입니다. Transformer에서는 query, key, value를 만들고 query-key 유사도로 attention weight를 계산한 뒤 value를 섞습니다.

이번 노트북의 목표는 다음과 같습니다.

- query, key, value의 역할을 구분합니다.
- scaled dot-product attention 계산 흐름을 직접 확인합니다.
- self-attention이 token 사이 정보를 섞는 방식을 시각화합니다.
- multi-head attention과 positional encoding이 왜 필요한지 이해합니다.

## 26-1. 준비

작은 행렬로 attention을 직접 계산합니다. 실제 Transformer 구현에서는 PyTorch가 이 계산을 효율적으로 처리하지만, 여기서는 NumPy로 흐름을 분해해서 봅니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)
np.random.seed(42)

## 26-2. Q, K, V의 직관

attention은 다음 질문으로 생각할 수 있습니다.

- Query: 현재 token이 무엇을 찾고 있는가?
- Key: 각 token은 어떤 특징을 가지고 있는가?
- Value: 실제로 섞어서 가져올 정보는 무엇인가?

query와 key의 유사도가 높으면 해당 value를 더 많이 참고합니다.

In [ ]:
tokens = ['CLS', 'P1', 'P2', 'P3']
X = np.array([
    [1.0, 0.2, 0.1],
    [0.9, 0.1, 0.3],
    [0.1, 1.0, 0.4],
    [0.2, 0.8, 0.9],
])

Wq = np.array([[0.8, 0.1], [0.2, 0.7], [0.3, 0.4]])
Wk = np.array([[0.7, 0.2], [0.1, 0.8], [0.4, 0.3]])
Wv = np.array([[0.6, 0.1], [0.2, 0.5], [0.1, 0.9]])

Q = X @ Wq
K = X @ Wk
V = X @ Wv

print('Q =')
print(Q)
print('\nK =')
print(K)
print('\nV =')
print(V)

## 26-3. Scaled dot-product attention

Transformer의 기본 attention은 다음 순서로 계산됩니다.

```text
score = QK^T / sqrt(d_k)
weight = softmax(score)
output = weight V
```

`sqrt(d_k)`로 나누는 이유는 차원이 커질수록 dot product 값이 너무 커져 softmax가 한쪽으로 심하게 치우치는 것을 줄이기 위해서입니다.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

d_k = K.shape[-1]
scores = Q @ K.T / np.sqrt(d_k)
weights = softmax(scores, axis=1)
attention_output = weights @ V

print('attention scores =')
print(scores)
print('\nattention weights =')
print(weights)
print('\nattention output =')
print(attention_output)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap='YlGnBu', vmin=0, vmax=weights.max())
ax.set_title('Self-attention weight')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel('참고되는 token')
ax.set_ylabel('현재 token')

for y in range(len(tokens)):
    for x in range(len(tokens)):
        ax.text(x, y, f'{weights[y, x]:.2f}', ha='center', va='center')

plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()

attention weight 행렬에서 한 행은 한 token이 다른 token들을 어떤 비율로 참고하는지 나타냅니다. self-attention에서는 query, key, value가 모두 같은 입력 token에서 만들어지므로, 입력 내부의 관계를 스스로 계산합니다.

## 26-4. Multi-head attention

하나의 attention head만 있으면 token 관계를 한 가지 관점으로만 봅니다. Multi-head attention은 여러 projection을 사용해 서로 다른 관계를 동시에 보게 합니다.

예를 들어 이미지 patch에서는 어떤 head는 색이나 질감의 유사성을, 다른 head는 위치적으로 떨어진 객체 부분의 관계를 볼 수 있습니다.

```text
head 1: local texture relation
head 2: object part relation
head 3: background-context relation
...
concat heads -> linear projection
```

In [ ]:
head_a = np.array([
    [0.55, 0.30, 0.10, 0.05],
    [0.25, 0.55, 0.15, 0.05],
    [0.10, 0.10, 0.60, 0.20],
    [0.05, 0.10, 0.25, 0.60],
])
head_b = np.array([
    [0.20, 0.20, 0.30, 0.30],
    [0.15, 0.25, 0.35, 0.25],
    [0.30, 0.30, 0.20, 0.20],
    [0.35, 0.25, 0.15, 0.25],
])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, data, title in zip(axes, [head_a, head_b], ['Head A: 가까운 token 중심', 'Head B: 넓은 문맥 중심']):
    ax.imshow(data, cmap='YlOrRd', vmin=0, vmax=0.6)
    ax.set_title(title)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_yticklabels(tokens)
    for y in range(len(tokens)):
        for x in range(len(tokens)):
            ax.text(x, y, f'{data[y, x]:.2f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 26-5. Positional encoding이 필요한 이유

self-attention은 token 집합의 관계를 계산하지만, 그 자체만으로는 token의 순서나 2D 위치를 강하게 알지 못합니다. 이미지에서는 patch가 어디에 있었는지가 중요하므로 위치 정보를 embedding에 더해야 합니다.

ViT에서는 보통 학습 가능한 positional embedding을 patch embedding에 더합니다.

```text
input token = patch embedding + positional embedding
```

In [ ]:
positions = np.arange(8)
dims = np.arange(4)
positional = np.zeros((len(positions), len(dims)))

for pos in positions:
    for i in range(0, len(dims), 2):
        positional[pos, i] = np.sin(pos / (10000 ** (i / len(dims))))
        positional[pos, i + 1] = np.cos(pos / (10000 ** (i / len(dims))))

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(positional, cmap='coolwarm', aspect='auto')
ax.set_title('예시 positional encoding')
ax.set_xlabel('embedding dimension')
ax.set_ylabel('position')
ax.set_xticks(range(len(dims)))
ax.set_yticks(range(len(positions)))
plt.show()

## 정리

- attention은 query-key 유사도로 value를 섞는 연산입니다.
- self-attention은 같은 입력 token들 사이의 관계를 계산합니다.
- multi-head attention은 여러 관계 관점을 동시에 학습하게 합니다.
- positional encoding은 Transformer가 token의 위치를 알 수 있게 해 줍니다.

다음 노트북 `27_Vision_Transformer_ViT_핵심_아이디어.ipynb`에서는 이미지 patch를 token으로 바꾸고 ViT 전체 구조를 연결합니다.